# Previsão Copa do Mundo 2026

Simulação estatística da Copa do Mundo FIFA 2026 (formato de 48 seleções) usando um modelo Poisson de gols treinado em histórico de partidas internacionais e ranking FIFA, combinado com simulação de Monte Carlo da fase de grupos e do mata-mata.

## 1. Carregamento e Tratamento Inicial dos Dados
Carregamos os dados e padronizamos os nomes históricos dos países para manter a continuidade estatística.

In [1]:
import pandas as pd
import numpy as np
import gc
import itertools
import random
from collections import Counter
np.random.seed(42)

In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson

In [3]:
data = 'data/data-csv/'

In [4]:
country_mapping = {
    'West Germany': 'Germany',
    'Czechoslovakia': 'Czech Republic',
    'Yugoslavia': 'Serbia',
    'Soviet Union': 'Russia',
    'Serbia and Montenegro': 'Serbia',
    'Dutch East Indies': 'Indonesia',
    'Zaire': 'DR Congo'
}

In [5]:
qualified_teams = pd.read_csv(data + 'qualified_teams.csv')
qualified_teams['team_name'] = qualified_teams['team_name'].replace(country_mapping)
qualified_teams.head()

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,count_matches,performance
0,1,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,5,final
1,2,WC-1930,1930 FIFA World Cup,T-06,Belgium,BEL,2,group stage
2,3,WC-1930,1930 FIFA World Cup,T-07,Bolivia,BOL,2,group stage
3,4,WC-1930,1930 FIFA World Cup,T-09,Brazil,BRA,2,group stage
4,5,WC-1930,1930 FIFA World Cup,T-13,Chile,CHL,3,group stage


## 2. Ranking FIFA (Elo) das Seleções
Carregamos o ranking FIFA, removemos colunas que não serão usadas no modelo e corrigimos divergências de nome entre esta base e a base de partidas.

### Fonte do Raking = https://www.kaggle.com/datasets/cashncarry/fifaworldranking/discussion/681592

In [6]:
elo_data = 'data/elo/fifa_ranking-2026.csv'
elo = pd.read_csv(elo_data)
elo

,Unnamed: 0,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,0,140.0,Brunei Darussalam,BRU,2.00,0.00,140,AFC,1992-12-31
1,1,33.0,Portugal,POR,38.00,0.00,33,UEFA,1992-12-31
2,2,32.0,Zambia,ZAM,38.00,0.00,32,CAF,1992-12-31
3,3,31.0,Greece,GRE,38.00,0.00,31,UEFA,1992-12-31
4,4,30.0,Algeria,ALG,39.00,0.00,30,CAF,1992-12-31
...,...,...,...,...,...,...,...,...,...
69987,69987,206.0,Bahamas,BAH,796.60,796.60,0,CONCACAF,2026-01-19
69988,69988,207.0,US Virgin Islands,VIR,776.60,776.60,0,CONCACAF,2026-01-19
69989,69989,208.0,British Virgin Islands,VGB,776.54,776.54,0,CONCACAF,2026-01-19
69990,69990,209.0,Anguilla,AIA,759.78,759.78,0,CONCACAF,2026-01-19


In [7]:
elo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69992 entries, 0 to 69991
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       69992 non-null  int64  
 1   rank             69983 non-null  float64
 2   country_full     69992 non-null  object 
 3   country_abrv     69992 non-null  object 
 4   total_points     69992 non-null  float64
 5   previous_points  69992 non-null  float64
 6   rank_change      69992 non-null  int64  
 7   confederation    69992 non-null  object 
 8   rank_date        69992 non-null  object 
dtypes: float64(3), int64(2), object(4)
memory usage: 4.8+ MB


In [8]:
elo = elo.drop(columns=['Unnamed: 0', 'country_abrv', 'confederation'])

In [9]:
elo['country_full'].unique()

array(['Brunei Darussalam', 'Portugal', 'Zambia', 'Greece', 'Algeria',
       'Yugoslavia', 'Wales', "Côte d'Ivoire", 'Austria', 'Bulgaria',
       'USA', 'Scotland', 'Cameroon', 'Egypt', 'Poland', 'France',
       'Czechoslovakia', 'Mexico', 'Colombia', 'Hungary', 'Costa Rica',
       'Zimbabwe', 'Malawi', 'Qatar', 'Senegal', 'El Salvador',
       'Korea Republic', 'Saudi Arabia', 'Chile', 'Iceland', 'Australia',
       'Finland', 'Northern Ireland', 'Türkiye', 'Morocco', 'Honduras',
       'Ghana', 'Tunisia', 'Belgium', 'Gabon', 'Uruguay', 'Nigeria',
       'The Gambia', 'Equatorial Guinea', 'Myanmar', 'Liechtenstein',
       'Nicaragua', 'Chad', 'Chinese Taipei', 'Nepal', 'Philippines',
       'Vanuatu', 'Aruba', 'Lebanon', 'Bahamas', 'Maldives', 'Rwanda',
       'Somalia', 'Namibia', 'Central African Republic', 'Seychelles',
       'Belarus', 'Switzerland', 'Romania', 'Argentina', 'Denmark',
       'Russia', 'Netherlands', 'Republic of Ireland', 'England',
       'Sweden', 'Brazil'

In [10]:
correcoes_nomes = {
    'USA': 'United States',
    "Côte d'Ivoire": 'Ivory Coast',
    'Korea Republic': 'South Korea',
    'Korea DPR': 'North Korea',
    'Türkiye': 'Turkey',
    'China PR': 'China',
    'Czechia': 'Czech Republic',
    'IR Iran': 'Iran',
    'Congo DR' : 'DR Congo',
    'Cabo Verde': 'Cape Verde'
}

elo['country_full'] = elo['country_full'].replace(correcoes_nomes)

In [11]:
elo['year'] = pd.to_datetime(elo['rank_date']).dt.year
elo['year']

0        1992
1        1992
2        1992
3        1992
4        1992
         ... 
69987    2026
69988    2026
69989    2026
69990    2026
69991    2026
Name: year, Length: 69992, dtype: int32

In [12]:
elo = elo.drop(columns=['previous_points', 'rank_change'])
elo

,rank,country_full,total_points,rank_date,year
0,140.0,Brunei Darussalam,2.00,1992-12-31,1992
1,33.0,Portugal,38.00,1992-12-31,1992
2,32.0,Zambia,38.00,1992-12-31,1992
3,31.0,Greece,38.00,1992-12-31,1992
4,30.0,Algeria,39.00,1992-12-31,1992
...,...,...,...,...,...
69987,206.0,Bahamas,796.60,2026-01-19,2026
69988,207.0,US Virgin Islands,776.60,2026-01-19,2026
69989,208.0,British Virgin Islands,776.54,2026-01-19,2026
69990,209.0,Anguilla,759.78,2026-01-19,2026


In [13]:
elo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69992 entries, 0 to 69991
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rank          69983 non-null  float64
 1   country_full  69992 non-null  object 
 2   total_points  69992 non-null  float64
 3   rank_date     69992 non-null  object 
 4   year          69992 non-null  int32  
dtypes: float64(2), int32(1), object(2)
memory usage: 2.4+ MB


## 3. Partidas Internacionais e Calendário da Copa 2026
Carregamos o histórico de partidas internacionais e separamos dele os 72 jogos da fase de grupos da Copa 2026, já cruzando cada seleção com seu Elo mais recente.

In [14]:
inter_matches_data = 'data/matches/'
inter_matches = pd.read_csv(inter_matches_data + 'results.csv')
inter_matches

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49373,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49374,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49375,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49376,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


In [15]:
inter_matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49378 entries, 0 to 49377
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49378 non-null  object 
 1   home_team   49378 non-null  object 
 2   away_team   49378 non-null  object 
 3   home_score  49306 non-null  float64
 4   away_score  49306 non-null  float64
 5   tournament  49378 non-null  object 
 6   city        49378 non-null  object 
 7   country     49378 non-null  object 
 8   neutral     49378 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


In [16]:
wc_2026_matches = inter_matches[inter_matches['date']>='2026-06-11']
wc_2026_matches

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49306,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49307,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
49308,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False
49309,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False
49310,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
...,...,...,...,...,...,...,...,...,...
49373,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49374,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49375,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49376,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


In [17]:
wc_2026_matches = wc_2026_matches.drop(columns=['date','home_score', 'away_score', 'tournament', 'city', 'country'])

In [18]:
wc_2026_matches['year'] = pd.to_datetime(inter_matches['date']).dt.year
wc_2026_matches

,home_team,away_team,neutral,year
49306,Mexico,South Africa,False,2026
49307,South Korea,Czech Republic,True,2026
49308,Canada,Bosnia and Herzegovina,False,2026
49309,United States,Paraguay,False,2026
49310,Qatar,Switzerland,True,2026
...,...,...,...,...
49373,Jordan,Argentina,True,2026
49374,Colombia,Portugal,True,2026
49375,DR Congo,Uzbekistan,True,2026
49376,Panama,England,True,2026


In [19]:
elo_atualizado = elo.drop_duplicates(subset=['country_full'], keep='last').copy()

wc_2026_matches = pd.merge(wc_2026_matches, elo_atualizado[['country_full', 'total_points']], left_on='home_team', right_on='country_full', how='left')

wc_2026_matches = pd.merge(wc_2026_matches, elo_atualizado[['country_full', 'total_points']], left_on='away_team', right_on='country_full', how='left')

wc_2026_matches = wc_2026_matches.drop(columns=['country_full_x', 'country_full_y'])

print(f"Total de jogos após o merge: {len(wc_2026_matches)}")

Total de jogos após o merge: 72


In [20]:
wc_2026_matches = wc_2026_matches.rename(columns={'total_points_x': 'home_elo', 'total_points_y': 'away_elo'})

In [21]:
wc_2026_matches['home_team'].unique()

array(['Mexico', 'South Korea', 'Canada', 'United States', 'Qatar',
       'Brazil', 'Haiti', 'Australia', 'Germany', 'Ivory Coast',
       'Netherlands', 'Sweden', 'Belgium', 'Iran', 'Spain',
       'Saudi Arabia', 'France', 'Iraq', 'Argentina', 'Austria',
       'Portugal', 'Uzbekistan', 'England', 'Ghana', 'Czech Republic',
       'Switzerland', 'Scotland', 'Turkey', 'Ecuador', 'Tunisia',
       'New Zealand', 'Uruguay', 'Norway', 'Jordan', 'Colombia', 'Panama',
       'South Africa', 'Bosnia and Herzegovina', 'Morocco', 'Paraguay',
       'Curaçao', 'Japan', 'Egypt', 'Cape Verde', 'Senegal', 'Algeria',
       'DR Congo', 'Croatia'], dtype=object)

## 4. Cruzamento do Histórico de Partidas com o Ranking
Filtramos o histórico de partidas a partir de 1992 e usamos merge_asof para anexar a cada jogo o Elo mais recente disponível de cada seleção até 90 dias antes da data da partida.


In [22]:
inter_matches_hist = inter_matches[inter_matches['date']<'2026-06-11']
inter_matches_hist

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49301,2026-06-03,Luxembourg,Italy,0.0,1.0,Friendly,Luxembourg,Luxembourg,False
49302,2026-06-03,Netherlands,Algeria,0.0,1.0,Friendly,Rotterdam,Netherlands,False
49303,2026-06-03,Panama,Dominican Republic,4.0,2.0,Friendly,Panama City,Panama,False
49304,2026-06-03,Poland,Nigeria,2.0,2.0,Friendly,Warsaw,Poland,False


In [23]:
inter_matches_hist = inter_matches_hist.drop(columns=['city', 'country'])
inter_matches_hist

,date,home_team,away_team,home_score,away_score,tournament,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False
...,...,...,...,...,...,...,...
49301,2026-06-03,Luxembourg,Italy,0.0,1.0,Friendly,False
49302,2026-06-03,Netherlands,Algeria,0.0,1.0,Friendly,False
49303,2026-06-03,Panama,Dominican Republic,4.0,2.0,Friendly,False
49304,2026-06-03,Poland,Nigeria,2.0,2.0,Friendly,False


In [24]:
inter_matches_hist['date'] = pd.to_datetime(inter_matches_hist['date'])

In [25]:
inter_matches_hist['year'] = inter_matches_hist['date'].dt.year
inter_matches_hist

,date,home_team,away_team,home_score,away_score,tournament,neutral,year
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False,1872
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False,1873
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False,1874
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False,1875
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False,1876
...,...,...,...,...,...,...,...,...
49301,2026-06-03,Luxembourg,Italy,0.0,1.0,Friendly,False,2026
49302,2026-06-03,Netherlands,Algeria,0.0,1.0,Friendly,False,2026
49303,2026-06-03,Panama,Dominican Republic,4.0,2.0,Friendly,False,2026
49304,2026-06-03,Poland,Nigeria,2.0,2.0,Friendly,False,2026


In [26]:
inter_matches_hist = inter_matches_hist[inter_matches_hist['year']>=1992]
inter_matches_hist

,date,home_team,away_team,home_score,away_score,tournament,neutral,year
18108,1992-01-04,Egypt,Czechoslovakia,2.0,0.0,Friendly,False,1992
18109,1992-01-05,DR Congo,Ivory Coast,2.0,0.0,Friendly,False,1992
18110,1992-01-05,Guyana,Barbados,0.0,2.0,Friendly,False,1992
18111,1992-01-07,Egypt,Norway,0.0,0.0,Friendly,False,1992
18112,1992-01-12,Cameroon,Morocco,1.0,0.0,African Cup of Nations,True,1992
...,...,...,...,...,...,...,...,...
49301,2026-06-03,Luxembourg,Italy,0.0,1.0,Friendly,False,2026
49302,2026-06-03,Netherlands,Algeria,0.0,1.0,Friendly,False,2026
49303,2026-06-03,Panama,Dominican Republic,4.0,2.0,Friendly,False,2026
49304,2026-06-03,Poland,Nigeria,2.0,2.0,Friendly,False,2026


In [27]:
inter_matches_hist['date'] = pd.to_datetime(inter_matches_hist['date'])
elo['rank_date'] = pd.to_datetime(elo['rank_date'])

elo_temporario = elo.rename(columns={'country_full': 'home_team'})

inter_matches_hist = inter_matches_hist.sort_values('date')
elo_temporario = elo_temporario.sort_values('rank_date')

inter_matches_hist = pd.merge_asof(
    inter_matches_hist,
    elo_temporario,
    left_on='date',
    right_on='rank_date',
    by='home_team',         
    direction='backward',  
    tolerance=pd.Timedelta(days=90)
)

inter_matches_hist = inter_matches_hist.rename(columns={'home_team': 'home_team', 'total_points': 'elo_home'})

inter_matches_hist

C:\Users\Arthur Melo\AppData\Local\Temp\ipykernel_11488\2300256864.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inter_matches_hist['date'] = pd.to_datetime(inter_matches_hist['date'])


,date,home_team,away_team,home_score,away_score,tournament,neutral,year_x,rank,elo_home,rank_date,year_y
0,1992-01-04,Egypt,Czechoslovakia,2.0,0.0,Friendly,False,1992,NaN,NaN,NaT,NaN
1,1992-01-05,DR Congo,Ivory Coast,2.0,0.0,Friendly,False,1992,NaN,NaN,NaT,NaN
2,1992-01-05,Guyana,Barbados,0.0,2.0,Friendly,False,1992,NaN,NaN,NaT,NaN
3,1992-01-07,Egypt,Norway,0.0,0.0,Friendly,False,1992,NaN,NaN,NaT,NaN
4,1992-01-12,Cameroon,Morocco,1.0,0.0,African Cup of Nations,True,1992,NaN,NaN,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
31193,2026-06-03,Gibraltar,British Virgin Islands,4.0,0.0,Friendly,False,2026,NaN,NaN,NaT,NaN
31194,2026-06-03,DR Congo,Denmark,0.0,0.0,Friendly,True,2026,NaN,NaN,NaT,NaN
31195,2026-06-03,Albania,Israel,0.0,1.0,Friendly,False,2026,NaN,NaN,NaT,NaN
31196,2026-06-03,South Korea,El Salvador,1.0,0.0,Friendly,True,2026,NaN,NaN,NaT,NaN


In [28]:
inter_matches_hist = inter_matches_hist.drop(columns=['year_y', 'rank_date', 'rank'])

In [29]:
elo_temporario = elo.rename(columns={'country_full': 'away_team'})

inter_matches_hist = pd.merge_asof(
    inter_matches_hist,
    elo_temporario,
    left_on='date',
    right_on='rank_date',
    by='away_team',         
    direction='backward',   
    tolerance=pd.Timedelta(days=90)
)

inter_matches_hist = inter_matches_hist.rename(columns={'home_team': 'home_team', 'total_points': 'elo_away'})

inter_matches_hist

,date,home_team,away_team,home_score,away_score,tournament,neutral,year_x,elo_home,rank,elo_away,rank_date,year
0,1992-01-04,Egypt,Czechoslovakia,2.0,0.0,Friendly,False,1992,NaN,NaN,NaN,NaT,NaN
1,1992-01-05,DR Congo,Ivory Coast,2.0,0.0,Friendly,False,1992,NaN,NaN,NaN,NaT,NaN
2,1992-01-05,Guyana,Barbados,0.0,2.0,Friendly,False,1992,NaN,NaN,NaN,NaT,NaN
3,1992-01-07,Egypt,Norway,0.0,0.0,Friendly,False,1992,NaN,NaN,NaN,NaT,NaN
4,1992-01-12,Cameroon,Morocco,1.0,0.0,African Cup of Nations,True,1992,NaN,NaN,NaN,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31193,2026-06-03,Gibraltar,British Virgin Islands,4.0,0.0,Friendly,False,2026,NaN,NaN,NaN,NaT,NaN
31194,2026-06-03,DR Congo,Denmark,0.0,0.0,Friendly,True,2026,NaN,NaN,NaN,NaT,NaN
31195,2026-06-03,Albania,Israel,0.0,1.0,Friendly,False,2026,NaN,NaN,NaN,NaT,NaN
31196,2026-06-03,South Korea,El Salvador,1.0,0.0,Friendly,True,2026,NaN,NaN,NaN,NaT,NaN


In [30]:
inter_matches_hist = inter_matches_hist.drop(columns=['year', 'rank_date', 'rank'])

In [31]:
rename_cols = {
    'elo_home': 'home_points',
    'elo_away': 'away_points',
    'year_x': 'year'
}

inter_matches_hist.rename(rename_cols, axis=1, inplace=True)
inter_matches_hist

,date,home_team,away_team,home_score,away_score,tournament,neutral,year,home_points,away_points
0,1992-01-04,Egypt,Czechoslovakia,2.0,0.0,Friendly,False,1992,NaN,NaN
1,1992-01-05,DR Congo,Ivory Coast,2.0,0.0,Friendly,False,1992,NaN,NaN
2,1992-01-05,Guyana,Barbados,0.0,2.0,Friendly,False,1992,NaN,NaN
3,1992-01-07,Egypt,Norway,0.0,0.0,Friendly,False,1992,NaN,NaN
4,1992-01-12,Cameroon,Morocco,1.0,0.0,African Cup of Nations,True,1992,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
31193,2026-06-03,Gibraltar,British Virgin Islands,4.0,0.0,Friendly,False,2026,NaN,NaN
31194,2026-06-03,DR Congo,Denmark,0.0,0.0,Friendly,True,2026,NaN,NaN
31195,2026-06-03,Albania,Israel,0.0,1.0,Friendly,False,2026,NaN,NaN
31196,2026-06-03,South Korea,El Salvador,1.0,0.0,Friendly,True,2026,NaN,NaN


## 5. Peso por Relevância do Torneio
Atribuímos um peso a cada partida conforme a importância do torneio em que ela ocorreu, para que jogos de Copa do Mundo pesem mais no treino do modelo do que amistosos.


In [32]:
df_matches = inter_matches_hist.copy()
df_matches = df_matches.dropna(subset=['home_points', 'away_points'])
df_matches

,date,home_team,away_team,home_score,away_score,tournament,neutral,year,home_points,away_points
600,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,True,1993,34.00,22.00
601,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,False,1993,27.00,11.00
602,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,False,1993,21.00,0.00
603,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,False,1993,27.00,34.00
604,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,True,1993,11.00,22.00
...,...,...,...,...,...,...,...,...,...,...
31145,2026-03-31,Morocco,Paraguay,2.0,1.0,Friendly,True,2026,1736.57,1501.50
31146,2026-03-31,Jordan,Nigeria,2.0,2.0,Friendly,True,2026,1388.93,1581.55
31147,2026-03-31,Ivory Coast,Scotland,1.0,0.0,Friendly,True,2026,1522.48,1506.77
31148,2026-03-31,Liberia,Libya,2.0,2.0,"Morocco, Capital of African Football",True,2026,1081.46,1183.06


In [33]:
df_matches.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26008 entries, 600 to 31149
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         26008 non-null  datetime64[ns]
 1   home_team    26008 non-null  object        
 2   away_team    26008 non-null  object        
 3   home_score   26008 non-null  float64       
 4   away_score   26008 non-null  float64       
 5   tournament   26008 non-null  object        
 6   neutral      26008 non-null  bool          
 7   year         26008 non-null  int32         
 8   home_points  26008 non-null  float64       
 9   away_points  26008 non-null  float64       
dtypes: bool(1), datetime64[ns](1), float64(4), int32(1), object(3)
memory usage: 1.9+ MB


In [34]:
df_matches['tournament'].unique()

array(['Friendly', 'African Cup of Nations qualification',
       'FIFA World Cup qualification', 'Nehru Cup', 'Merdeka Tournament',
       'UNCAF Cup', 'Four Nations Tournament',
       'CONMEBOL–UEFA Cup of Champions', 'Copa Paz del Chaco',
       'Kirin Cup', 'CFU Caribbean Cup qualification',
       'Indian Ocean Island Games', 'Amílcar Cabral Cup',
       'South Pacific Mini Games', 'South Asian Games',
       'United Arab Emirates Friendship Tournament',
       'Malta International Tournament', 'Lunar New Year Cup',
       'Joe Robbie Cup', 'African Cup of Nations', 'CFU Caribbean Cup',
       'UEFA Euro qualification', 'Miami Cup', 'FIFA World Cup',
       'Melanesia Cup', 'Baltic Cup', 'Asian Games', 'Gulf Cup',
       'Simba Tournament', 'CECAFA Cup', 'Confederations Cup',
       'Dynasty Cup', "King's Cup", 'SAFF Cup', 'Korea Cup', 'USA Cup',
       'Copa América', 'South Pacific Games', 'Trans-Tasman Cup',
       'Oceania Nations Cup', 'Windward Islands Tournament',
       '

In [35]:
pesos_torneios = {
    # Nível 1: O Topo Absoluto (Peso 3.0)
    'FIFA World Cup': 3.0,

    # Nível 2: Copas Continentais Principais (Peso 2.5)
    'UEFA Euro': 2.5,
    'Copa América': 2.5,
    'African Cup of Nations': 2.5,
    'AFC Asian Cup': 2.5,
    'Gold Cup': 2.5,
    'Oceania Nations Cup': 2.5,

    # Nível 3: Eliminatórias de Alto Nível e Finais Intercontinentais (Peso 2.0)
    'FIFA World Cup qualification': 2.0,
    'UEFA Euro qualification': 2.0,
    'Copa América qualification': 2.0,
    'African Cup of Nations qualification': 2.0,
    'AFC Asian Cup qualification': 2.0,
    'Gold Cup qualification': 2.0,
    'Oceania Nations Cup qualification': 2.0,
    'CONMEBOL–UEFA Cup of Champions': 2.0,

    # Nível 4: Torneios Secundários e Ligas de Nações (Peso 1.5)
    'Confederations Cup': 1.5,
    'UEFA Nations League': 1.5,
    'CONCACAF Nations League': 1.5,
    'CONCACAF Nations League qualification': 1.5,
    'Intercontinental Cup': 1.5,
    'EAFF Championship': 1.5,
    'WAFF Championship': 1.5,
    'AFF Championship': 1.5,
    'ASEAN Championship': 1.5,
    'SAFF Cup': 1.5,
    'CAFA Nations Cup': 1.5,
    'COSAFA Cup': 1.5,
    'CFU Caribbean Cup': 1.5,
}

In [36]:
df_matches['tournament'] = df_matches['tournament'].map(pesos_torneios).fillna(1.0)
df_matches

,date,home_team,away_team,home_score,away_score,tournament,neutral,year,home_points,away_points
600,1993-01-01,Ghana,Mali,1.0,1.0,1.0,True,1993,34.00,22.00
601,1993-01-02,Gabon,Burkina Faso,1.0,1.0,1.0,False,1993,27.00,11.00
602,1993-01-02,Kuwait,Lebanon,2.0,0.0,1.0,False,1993,21.00,0.00
603,1993-01-03,Gabon,Ghana,2.0,3.0,1.0,False,1993,27.00,34.00
604,1993-01-03,Burkina Faso,Mali,1.0,0.0,1.0,True,1993,11.00,22.00
...,...,...,...,...,...,...,...,...,...,...
31145,2026-03-31,Morocco,Paraguay,2.0,1.0,1.0,True,2026,1736.57,1501.50
31146,2026-03-31,Jordan,Nigeria,2.0,2.0,1.0,True,2026,1388.93,1581.55
31147,2026-03-31,Ivory Coast,Scotland,1.0,0.0,1.0,True,2026,1522.48,1506.77
31148,2026-03-31,Liberia,Libya,2.0,2.0,1.0,True,2026,1081.46,1183.06


## 6. Filtro do Universo de Treino
Restringimos a base a partidas a partir de 2018. Isso porque partidas muito antigas não interverem no peso atual da seleção. Por exemplo, o Brasil de 1970 não diz nada sobre o Brasil de 2026.

In [37]:
df_matches = df_matches[df_matches['year'] >=2018]

In [38]:
df_matches.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6742 entries, 23189 to 31149
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         6742 non-null   datetime64[ns]
 1   home_team    6742 non-null   object        
 2   away_team    6742 non-null   object        
 3   home_score   6742 non-null   float64       
 4   away_score   6742 non-null   float64       
 5   tournament   6742 non-null   float64       
 6   neutral      6742 non-null   bool          
 7   year         6742 non-null   int32         
 8   home_points  6742 non-null   float64       
 9   away_points  6742 non-null   float64       
dtypes: bool(1), datetime64[ns](1), float64(5), int32(1), object(2)
memory usage: 507.0+ KB


## 7. Decaimento Temporal
$$
W_{final} = W_{torneio} \times e^{-\alpha \cdot t}$$
Onde:

$W_{final}$ é o peso final daquela linha no modelo.

$W_{torneio}$ é o peso da competição df['tournaments'].

$t$ é o tempo decorrido em anos entre o jogo e a data de hoje (2026).

$\alpha$ (alfa) é a taxa de esquecimento. Usamos $\alpha = 0.001265$ para que a importância do jogo caia pela metade a cada ano (tempo de meia vida de 1.5 anos).

### 7.1 Decaimento Temporal Ideal

In [39]:
DATA_CORTE = pd.to_datetime('2022-11-20')
meias_vidas_anos = [1, 1.5, 2, 2.5, 3, 4, 5]

df_teste = df_matches[
    (df_matches['date'] >= DATA_CORTE) & 
    (df_matches['date'] <= pd.to_datetime('2022-12-18')) & 
    (df_matches['tournament'] == 3.0) 
].copy()

df_treino_base = df_matches[df_matches['date'] < DATA_CORTE].copy()
df_treino_base['dias_atras'] = (DATA_CORTE - df_treino_base['date']).dt.days

resultados_validacao = []

def calcular_probabilidades_1x2(lambda_home, lambda_away):
    max_gols = 10
    prob_home_win, prob_draw, prob_away_win = 0.0, 0.0, 0.0
    for h in range(max_gols):
        for a in range(max_gols):
            prob = poisson.pmf(h, lambda_home) * poisson.pmf(a, lambda_away)
            if h > a:
                prob_home_win += prob
            elif h == a:
                prob_draw += prob
            else:
                prob_away_win += prob
    return prob_home_win, prob_draw, prob_away_win

print("Iniciando validação do hiperparâmetro Alfa (Meia-vida)...")

for mv in meias_vidas_anos:
    alpha = np.log(2) / (mv * 365.25)
    
    df_treino = df_treino_base.copy()
    df_treino['decaimento'] = df_treino['tournament'] * np.exp(-alpha * df_treino['dias_atras'])
    
    df_home = df_treino[['home_team', 'away_team', 'home_score', 'home_points', 'away_points', 'neutral', 'decaimento']].copy()
    df_home.columns = ['time', 'adversario', 'gols', 'elo_time', 'elo_adversario', 'neutro', 'peso']
    df_home['mando_de_campo'] = 1

    df_away = df_treino[['away_team', 'home_team', 'away_score', 'away_points', 'home_points', 'neutral', 'decaimento']].copy()
    df_away.columns = ['time', 'adversario', 'gols', 'elo_time', 'elo_adversario', 'neutro', 'peso']
    df_away['mando_de_campo'] = 0

    df_modelo_val = pd.concat([df_home, df_away], ignore_index=True)
    df_modelo_val.loc[df_modelo_val['neutro'] == True, 'mando_de_campo'] = 0
    
    df_modelo_val['time'] = df_modelo_val['time'].astype('category')
    df_modelo_val['adversario'] = df_modelo_val['adversario'].astype('category')

    formula_poisson = "gols ~ elo_time + elo_adversario + mando_de_campo"
    
    modelo_val = smf.glm(
        formula=formula_poisson,
        data=df_modelo_val,
        family=sm.families.Poisson(),
        freq_weights=df_modelo_val['peso']
    ).fit()
    
    brier_scores = []
    
    for _, row in df_teste.iterrows():
        ataque_h = pd.DataFrame({
            'time': [row['home_team']], 'adversario': [row['away_team']], 
            'mando_de_campo': [0], 'elo_time': [row['home_points']], 'elo_adversario': [row['away_points']]
        })
        ataque_a = pd.DataFrame({
            'time': [row['away_team']], 'adversario': [row['home_team']], 
            'mando_de_campo': [0], 'elo_time': [row['away_points']], 'elo_adversario': [row['home_points']]
        })
        
        try:
            lambda_h = modelo_val.predict(ataque_h).iloc[0]
            lambda_a = modelo_val.predict(ataque_a).iloc[0]
        except Exception:
            continue
        
        ph, pdx, pa = calcular_probabilidades_1x2(lambda_h, lambda_a)
        
        real_h = 1 if row['home_score'] > row['away_score'] else 0
        real_d = 1 if row['home_score'] == row['away_score'] else 0
        real_a = 1 if row['home_score'] < row['away_score'] else 0
        
        brier = (ph - real_h)**2 + (pdx - real_d)**2 + (pa - real_a)**2
        brier_scores.append(brier)

    resultados_validacao.append({
        'Meia-Vida (Anos)': mv,
        'Alpha': alpha,
        'Brier Score': np.mean(brier_scores)
    })

df_resultados = pd.DataFrame(resultados_validacao).sort_values(by='Brier Score')
print("\nResultados da Validação de Hiperparâmetro (Copa 2022)")
print(df_resultados.to_string(index=False))

melhor_alpha = df_resultados.iloc[0]['Alpha']
melhor_mv = df_resultados.iloc[0]['Meia-Vida (Anos)']
print(f"\nMelhor hiperparâmetro escolhido para o modelo final: Meia-Vida de {melhor_mv} anos (Alpha: {melhor_alpha:.6f})")

Iniciando validação do hiperparâmetro Alfa (Meia-vida)...

Resultados da Validação de Hiperparâmetro (Copa 2022)
 Meia-Vida (Anos)    Alpha  Brier Score
              1.5 0.001265     0.586617
              2.0 0.000949     0.586660
              1.0 0.001898     0.586664
              2.5 0.000759     0.586717
              3.0 0.000633     0.586771
              4.0 0.000474     0.586857
              5.0 0.000380     0.586920

Melhor hiperparâmetro escolhido para o modelo final: Meia-Vida de 1.5 anos (Alpha: 0.001265)


In [40]:
df_matches = df_matches.copy()

df_matches['date'] = pd.to_datetime(df_matches['date'])
data_limite = pd.to_datetime('2026-06-11')

df_matches['dias_atras'] = (data_limite - df_matches['date']).dt.days

alpha_otimizado = melhor_alpha
df_matches['decaimento'] = df_matches['tournament'] * np.exp(-alpha_otimizado * df_matches['dias_atras'])
df_matches

,date,home_team,away_team,home_score,away_score,tournament,neutral,year,home_points,away_points,dias_atras,decaimento
23189,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,1.0,True,2018,438.00,476.00,3082,0.020258
23190,2018-01-02,Oman,Bahrain,1.0,0.0,1.0,True,2018,351.00,282.00,3082,0.020258
23191,2018-01-05,Oman,United Arab Emirates,0.0,0.0,1.0,True,2018,351.00,476.00,3079,0.020335
23192,2018-01-07,Estonia,Sweden,1.0,1.0,1.0,True,2018,397.00,998.00,3077,0.020386
23193,2018-01-11,Denmark,Sweden,0.0,1.0,1.0,True,2018,1099.00,998.00,3073,0.020490
...,...,...,...,...,...,...,...,...,...,...,...,...
31145,2026-03-31,Morocco,Paraguay,2.0,1.0,1.0,True,2026,1736.57,1501.50,72,0.912934
31146,2026-03-31,Jordan,Nigeria,2.0,2.0,1.0,True,2026,1388.93,1581.55,72,0.912934
31147,2026-03-31,Ivory Coast,Scotland,1.0,0.0,1.0,True,2026,1522.48,1506.77,72,0.912934
31148,2026-03-31,Liberia,Libya,2.0,2.0,1.0,True,2026,1081.46,1183.06,72,0.912934


## 8. Filtro de Seleções Relevantes e Preparação da Base do Modelo
Mantemos apenas partidas que envolvem ao menos uma seleção classificada para 2026 e transformamos cada jogo em duas linhas (mandante e visitante), montando a base final usada para treinar o modelo de gols.

In [41]:
df_matches.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'year', 'home_points', 'away_points',
       'dias_atras', 'decaimento'],
      dtype='object')

In [42]:
wc_2026_teams = pd.concat([
    wc_2026_matches['home_team'],
    wc_2026_matches['away_team']
]).unique()

df_matches = df_matches[df_matches['home_team'].isin(wc_2026_teams) | df_matches['away_team'].isin(wc_2026_teams)]
df_matches

,date,home_team,away_team,home_score,away_score,tournament,neutral,year,home_points,away_points,dias_atras,decaimento
23189,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,1.0,True,2018,438.00,476.00,3082,0.020258
23192,2018-01-07,Estonia,Sweden,1.0,1.0,1.0,True,2018,397.00,998.00,3077,0.020386
23193,2018-01-11,Denmark,Sweden,0.0,1.0,1.0,True,2018,1099.00,998.00,3073,0.020490
23195,2018-01-11,Jordan,Finland,1.0,2.0,1.0,True,2018,313.00,531.00,3073,0.020490
23197,2018-01-27,Moldova,South Korea,0.0,1.0,1.0,True,2018,112.00,569.00,3057,0.020909
...,...,...,...,...,...,...,...,...,...,...,...,...
31141,2026-03-31,Haiti,Iceland,1.0,1.0,1.0,True,2026,1294.49,1344.72,72,0.912934
31143,2026-03-31,Mexico,Belgium,1.0,1.0,1.0,True,2026,1675.75,1730.71,72,0.912934
31145,2026-03-31,Morocco,Paraguay,2.0,1.0,1.0,True,2026,1736.57,1501.50,72,0.912934
31146,2026-03-31,Jordan,Nigeria,2.0,2.0,1.0,True,2026,1388.93,1581.55,72,0.912934


In [43]:
df_matches.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3399 entries, 23189 to 31147
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         3399 non-null   datetime64[ns]
 1   home_team    3399 non-null   object        
 2   away_team    3399 non-null   object        
 3   home_score   3399 non-null   float64       
 4   away_score   3399 non-null   float64       
 5   tournament   3399 non-null   float64       
 6   neutral      3399 non-null   bool          
 7   year         3399 non-null   int32         
 8   home_points  3399 non-null   float64       
 9   away_points  3399 non-null   float64       
 10  dias_atras   3399 non-null   int64         
 11  decaimento   3399 non-null   float64       
dtypes: bool(1), datetime64[ns](1), float64(6), int32(1), int64(1), object(2)
memory usage: 308.7+ KB


In [44]:
df_home = df_matches[['home_team', 'away_team', 'home_score', 'home_points', 'away_points', 'neutral', 'decaimento']].copy()
df_home.columns = ['time', 'adversario', 'gols', 'elo_time', 'elo_adversario', 'neutro', 'peso']
df_home['mando_de_campo'] = 1

df_away = df_matches[['away_team', 'home_team', 'away_score', 'away_points', 'home_points','neutral', 'decaimento']].copy()
df_away.columns = ['time', 'adversario', 'gols', 'elo_time', 'elo_adversario', 'neutro', 'peso']
df_away['mando_de_campo'] = 0

df_modelo = pd.concat([df_home, df_away], ignore_index=True)
df_modelo.loc[df_modelo['neutro'] == True, 'mando_de_campo'] = 0

In [45]:
del df_matches, df_home, df_away
gc.collect()

17

In [46]:
df_modelo['time'] = df_modelo['time'].astype('category')
df_modelo['adversario'] = df_modelo['adversario'].astype('category')

In [47]:
df_modelo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6798 entries, 0 to 6797
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   time            6798 non-null   category
 1   adversario      6798 non-null   category
 2   gols            6798 non-null   float64 
 3   elo_time        6798 non-null   float64 
 4   elo_adversario  6798 non-null   float64 
 5   neutro          6798 non-null   bool    
 6   peso            6798 non-null   float64 
 7   mando_de_campo  6798 non-null   int64   
dtypes: bool(1), category(2), float64(4), int64(1)
memory usage: 310.0 KB


## 9. Ajuste do Modelo Poisson
Treinamos um modelo linear generalizado (GLM) Poisson que estima os gols esperados de uma seleção em função do próprio Elo, do Elo do adversário e do mando de campo, e validamos o resultado com um confronto de teste entre Brasil e Argentina.

In [48]:
formula_poisson = "gols ~ elo_time + elo_adversario + mando_de_campo"

modelo = smf.glm(
    formula=formula_poisson,
    data=df_modelo,
    family=sm.families.Poisson(),
    freq_weights=df_modelo['peso']
).fit()

print(modelo.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   gols   No. Observations:                 6798
Model:                            GLM   Df Residuals:                  2846.79
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4020.6
Date:                Wed, 01 Jul 2026   Deviance:                       3340.3
Time:                        22:25:16   Pearson chi2:                 3.05e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1812
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.0977      0.152      0.

In [174]:
def testar_brasil_argentina(modelo):
    # Simulando Brasil vs Argentina em Copa do Mundo
    brasil_elo = elo[elo['country_full'] == 'Brazil']['total_points'].values[0]
    argentina_elo = elo[elo['country_full'] == 'Argentina']['total_points'].values[0]
    
    ataque_br = pd.DataFrame({
        'time': ['Brazil'], 
        'adversario': ['Argentina'], 
        'mando_de_campo': [0], 
        'elo_time': brasil_elo,
        'elo_adversario': argentina_elo
    })
    
    ataque_arg = pd.DataFrame({
        'time': ['Argentina'], 
        'adversario': ['Brazil'], 
        'mando_de_campo': [0], 
        'elo_time': argentina_elo,
        'elo_adversario': brasil_elo
    })
    
    lambda_br = modelo.predict(ataque_br).iloc[0]
    lambda_arg = modelo.predict(ataque_arg).iloc[0]
    
    print(f"Gols Esperados (Matemática): Brasil {lambda_br:.2f} x {lambda_arg:.2f} Argentina")
    
    print("\n--- Sorteando 5 simulações deste jogo ---")
    for i in range(10):
        gols_br = np.random.poisson(lambda_br)
        gols_arg = np.random.poisson(lambda_arg)
        print(f"Simulação {i+1}: Brasil {gols_br} x {gols_arg} Argentina")

testar_brasil_argentina(modelo)

Gols Esperados (Matemática): Brasil 1.11 x 1.09 Argentina

--- Sorteando 5 simulações deste jogo ---
Simulação 1: Brasil 2 x 1 Argentina
Simulação 2: Brasil 0 x 0 Argentina
Simulação 3: Brasil 3 x 2 Argentina
Simulação 4: Brasil 0 x 0 Argentina
Simulação 5: Brasil 0 x 1 Argentina
Simulação 6: Brasil 0 x 1 Argentina
Simulação 7: Brasil 0 x 1 Argentina
Simulação 8: Brasil 1 x 1 Argentina
Simulação 9: Brasil 0 x 1 Argentina
Simulação 10: Brasil 0 x 3 Argentina


## 10. Grupos e Calendário da Copa 2026
Definimos os 12 grupos de 4 seleções da Copa 2026 e associamos cada um dos 72 jogos da fase de grupos ao seu respectivo grupo.

In [49]:
dicionario_grupos_2026 = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Haiti', 'Scotland', 'Morocco'],
    'D': ['United States', 'Paraguay', 'Australia','Turkey'],
    'E': ['Curaçao', 'Ecuador', 'Ivory Coast', 'Germany'],
    'F': ['Tunisia', 'Netherlands', 'Sweden', 'Japan'],
    'G': ['Belgium','New Zealand','Iran', 'Egypt'],
    'H': ['Spain','Saudi Arabia', 'Uruguay', 'Cape Verde'],
    'I': ['France','Iraq','Senegal', 'Norway'],
    'J': ['Argentina', 'Austria', 'Jordan', 'Algeria'],
    'K': ['Colombia','Portugal', 'Uzbekistan', 'DR Congo'],
    'L': ['England', 'Croatia', 'Ghana','Panama']
}

In [50]:
wc_2026_matches['group'] = wc_2026_matches.apply(lambda row: next((grupo for grupo, times in dicionario_grupos_2026.items() if row['home_team'] in times and row['away_team'] in times), None), axis=1)

In [51]:
wc_2026_matches

,home_team,away_team,neutral,year,home_elo,away_elo,group
0,Mexico,South Africa,False,2026,1675.75,1432.76,A
1,South Korea,Czech Republic,True,2026,1599.45,1487.00,A
2,Canada,Bosnia and Herzegovina,False,2026,1559.15,1362.37,B
3,United States,Paraguay,False,2026,1681.88,1501.50,D
4,Qatar,Switzerland,True,2026,1454.96,1654.69,B
...,...,...,...,...,...,...,...
67,Jordan,Argentina,True,2026,1388.93,1873.33,J
68,Colombia,Portugal,True,2026,1701.30,1760.38,K
69,DR Congo,Uzbekistan,True,2026,1468.22,1462.03,K
70,Panama,England,True,2026,1539.47,1834.12,L


## 11. Simulação de uma Rodada de Jogos
Criamos a função que simula o placar de uma partida a partir dos gols esperados pelo modelo e usamos uma única rodada de simulação para montar a classificação da fase de grupos e identificar os 32 classificados ao mata-mata.

In [52]:
def simular_partida(time_A, time_B, elo_A, elo_B, modelo):
    df_ataque_A = pd.DataFrame({
        'time': [time_A], 
        'adversario': [time_B], 
        'mando_de_campo': [0],
        'elo_time': elo_A,
        'elo_adversario': elo_B
    })
    
    df_ataque_B = pd.DataFrame({
        'time': [time_B], 
        'adversario': [time_A], 
        'mando_de_campo': [0], 
        'elo_time': elo_B,
        'elo_adversario': elo_A
    })
    
    lambda_A = modelo.predict(df_ataque_A).iloc[0]
    lambda_B = modelo.predict(df_ataque_B).iloc[0]
    
    gols_A = np.random.poisson(lambda_A)
    gols_B = np.random.poisson(lambda_B)
    
    return gols_A, gols_B

In [53]:
gols_casa = []
gols_fora = []

print("Simulando os 72 jogos da Fase de Grupos da Copa de 2026...\n")

for index, row in wc_2026_matches.iterrows():
    time_A = row['home_team']
    time_B = row['away_team']
    elo_A = row['home_elo']
    elo_B = row['away_elo']
    
    gols_A, gols_B = simular_partida(time_A, time_B, elo_A, elo_B, modelo)
    
    gols_casa.append(gols_A)
    gols_fora.append(gols_B)

wc_2026_matches['sim_gols_casa'] = gols_casa
wc_2026_matches['sim_gols_fora'] = gols_fora

print(wc_2026_matches[['home_team', 'sim_gols_casa', 'sim_gols_fora', 'away_team', 'group']].head(10))

Simulando os 72 jogos da Fase de Grupos da Copa de 2026...

       home_team  sim_gols_casa  sim_gols_fora               away_team group
0         Mexico              3              0            South Africa     A
1    South Korea              0              0          Czech Republic     A
2         Canada              3              2  Bosnia and Herzegovina     B
3  United States              0              0                Paraguay     D
4          Qatar              0              2             Switzerland     B
5         Brazil              1              0                 Morocco     C
6          Haiti              0              2                Scotland     C
7      Australia              1              0                  Turkey     D
8        Germany              2              3                 Curaçao     E
9    Ivory Coast              0              1                 Ecuador     E


In [54]:
tabelas_grupos = []

for grupo, jogos in wc_2026_matches.groupby('group'):
    times_do_grupo = dicionario_grupos_2026[grupo]
    
    tabela = {time: {'Grupo': grupo, 'Pts': 0, 'J': 0, 'V': 0, 'E': 0, 'D': 0, 'GM': 0, 'GS': 0, 'SG': 0} for time in times_do_grupo}
    
    for index, row in jogos.iterrows():
        tA = row['home_team']
        tB = row['away_team']
        gA = row['sim_gols_casa']
        gB = row['sim_gols_fora']
        
        tabela[tA]['J'] += 1
        tabela[tB]['J'] += 1
        tabela[tA]['GM'] += gA
        tabela[tB]['GM'] += gB
        tabela[tA]['GS'] += gB
        tabela[tB]['GS'] += gA
        
        if gA > gB:
            tabela[tA]['Pts'] += 3
            tabela[tA]['V'] += 1
            tabela[tB]['D'] += 1
        elif gA < gB:
            tabela[tB]['Pts'] += 3
            tabela[tB]['V'] += 1
            tabela[tA]['D'] += 1
        else:
            tabela[tA]['Pts'] += 1
            tabela[tB]['Pts'] += 1
            tabela[tA]['E'] += 1
            tabela[tB]['E'] += 1

    for time in times_do_grupo:
        tabela[time]['SG'] = tabela[time]['GM'] - tabela[time]['GS']
        
    df_grupo = pd.DataFrame.from_dict(tabela, orient='index')
    df_grupo = df_grupo.sort_values(by=['Pts', 'SG', 'GM'], ascending=[False, False, False])
    
    df_grupo['Posicao'] = range(1, 5)
    df_grupo['Time'] = df_grupo.index
    
    tabelas_grupos.append(df_grupo)

df_classificacao_final = pd.concat(tabelas_grupos, ignore_index=True)

print("\n--- CLASSIFICAÇÃO FINAL DA FASE DE GRUPOS ---")
print(df_classificacao_final[['Grupo', 'Posicao', 'Time', 'Pts', 'SG', 'GM']].head(12))


--- CLASSIFICAÇÃO FINAL DA FASE DE GRUPOS ---
   Grupo  Posicao                    Time  Pts  SG  GM
0      A        1                  Mexico    7   4   6
1      A        2          Czech Republic    5   1   4
2      A        3            South Africa    3  -3   2
3      A        4             South Korea    1  -2   0
4      B        1                  Canada    6   2   7
5      B        2  Bosnia and Herzegovina    4   2   5
6      B        3             Switzerland    4   0   3
7      B        4                   Qatar    3  -4   2
8      C        1                  Brazil    7   3   3
9      C        2                 Morocco    6   1   2
10     C        3                Scotland    3  -1   2
11     C        4                   Haiti    1  -3   0


In [55]:
classificados_diretos = df_classificacao_final[df_classificacao_final['Posicao'].isin([1, 2])]

terceiros_colocados = df_classificacao_final[df_classificacao_final['Posicao'] == 3]

melhores_terceiros = terceiros_colocados.sort_values(
    by=['Pts', 'SG', 'GM'], 
    ascending=[False, False, False]
).head(8)

times_mata_mata = pd.concat([classificados_diretos, melhores_terceiros]).sort_values(by=['Posicao'])

print(times_mata_mata[['Grupo', 'Posicao', 'Time', 'Pts', 'SG', 'GM']])

   Grupo  Posicao                    Time  Pts  SG  GM
0      A        1                  Mexico    7   4   6
4      B        1                  Canada    6   2   7
44     L        1                 England    6   2   5
8      C        1                  Brazil    7   3   3
12     D        1               Australia    7   4   5
40     K        1                DR Congo    7   2   3
16     E        1                 Ecuador    6   3   4
20     F        1                   Japan    9   4   4
36     J        1                 Algeria    9   6   7
24     G        1                    Iran    9   3   4
28     H        1                 Uruguay    6   1   3
32     I        1                  France    9   6   6
45     L        2                 Croatia    4   0   2
41     K        2              Uzbekistan    5   1   2
37     J        2               Argentina    6   5   9
33     I        2                 Senegal    6   2   4
29     H        2                   Spain    5   2   2
25     G  

## 12. Montagem do Mata-Mata
Criamos as funções que decidem o vencedor de um confronto eliminatório, montam o chaveamento das oitavas de final a partir da classificação da fase de grupos e avançam os times de uma fase para a outra. 

In [56]:
def simular_jogo_matamata(time_A, time_B, elo_A, elo_B, modelo):
    gols_A, gols_B = simular_partida(time_A, time_B, elo_A, elo_B, modelo)
    
    if gols_A > gols_B:
        return time_A
    elif gols_B > gols_A:
        return time_B
    else:
        return random.choice([time_A, time_B])

In [57]:
def montar_chave_oficial_2026(df_mata_mata):
    p = df_mata_mata[df_mata_mata['Posicao'] == 1].set_index('Grupo')['Time'].to_dict()
    s = df_mata_mata[df_mata_mata['Posicao'] == 2].set_index('Grupo')['Time'].to_dict()
    t = df_mata_mata[df_mata_mata['Posicao'] == 3][['Grupo', 'Time']].values.tolist()
    
    chaveamento = []
    
    def buscar_terceiro(grupos_permitidos):
        for i, (grupo_do_time, nome_do_time) in enumerate(t):
            if grupo_do_time in grupos_permitidos:
                return t.pop(i)[1] 
                

        if len(t) > 0:
            return t.pop(0)[1]
        return "Time Fantasma" 

    
    # Lado Esquerdo da Chave
    chaveamento.extend([p['E'], buscar_terceiro(['A', 'B', 'C', 'D', 'F'])]) # 1º E
    chaveamento.extend([p['I'], buscar_terceiro(['C', 'D', 'E', 'F', 'H'])]) # 1º I

    chaveamento.extend([s['A'], s['B']]) # Fixo
    chaveamento.extend([p['F'], s['C']]) # Fixo

    chaveamento.extend([s['K'], s['L']]) # Fixo
    chaveamento.extend([p['H'], s['J']]) # Fixo

    chaveamento.extend([p['D'], buscar_terceiro(['B', 'E', 'F', 'I', 'J'])]) # 1º D
    chaveamento.extend([p['G'], buscar_terceiro(['A', 'E', 'H', 'I', 'J'])]) # 1º G


    # Lado Direito da Chave
    chaveamento.extend([p['C'], s['F']]) # Fixo
    chaveamento.extend([s['E'], s['I']]) # Fixo

    chaveamento.extend([p['A'], buscar_terceiro(['C', 'E', 'F', 'H', 'I'])]) # 1º A
    chaveamento.extend([p['L'], buscar_terceiro(['E', 'H', 'I', 'J', 'K'])]) # 1º L
    
    chaveamento.extend([p['J'], s['H']]) # Fixo
    chaveamento.extend([s['D'], s['G']]) # Fixo
    
    
    chaveamento.extend([p['B'], buscar_terceiro(['E', 'F', 'G', 'I', 'J'])]) # 1º B
    chaveamento.extend([p['K'], buscar_terceiro(['D', 'E', 'I', 'J', 'L'])]) # 1º K
    
    
    return chaveamento

In [58]:
def jogar_fase(times_ativos, dicionario_elos, modelo):
    avancam = []
    for i in range(0, len(times_ativos), 2):
        tA = times_ativos[i]
        tB = times_ativos[i+1]
        
        elo_A = dicionario_elos.get(tA, 1500) 
        elo_B = dicionario_elos.get(tB, 1500)
        
        vencedor = simular_jogo_matamata(tA, tB, elo_A, elo_B, modelo)
        avancam.append(vencedor)
        
    return avancam

## 13. Simulação de Monte Carlo da Copa Completa
Repetimos a fase de grupos e o mata-mata 100.000 vezes, recalculando a classificação e o chaveamento a cada simulação, para estimar a probabilidade de título de cada seleção.

In [ ]:
# MONTE CARLO
lista_32_times = times_mata_mata['Time'].tolist()
dicionario_elos = dict(zip(elo_atualizado['country_full'], elo_atualizado['total_points']))

campeoes = []
simulacoes = 100000

df_partidas_A = wc_2026_matches[['home_team', 'away_team', 'home_elo', 'away_elo']].copy()
df_partidas_A.columns = ['time', 'adversario', 'elo_time', 'elo_adversario']
df_partidas_A['mando_de_campo'] = 0

df_partidas_B = wc_2026_matches[['away_team', 'home_team', 'away_elo', 'home_elo']].copy()
df_partidas_B.columns = ['time', 'adversario', 'elo_time', 'elo_adversario']
df_partidas_B['mando_de_campo'] = 0

print(f"Iniciando Monte Carlo: Simulando a Copa {simulacoes} vezes...")

for i in range(simulacoes):
    lambdas_A = modelo.predict(df_partidas_A)
    lambdas_B = modelo.predict(df_partidas_B)

    wc_2026_matches['sim_gols_casa'] = np.random.poisson(lambdas_A)
    wc_2026_matches['sim_gols_fora'] = np.random.poisson(lambdas_B)

    tabelas_grupos = []

    for grupo, jogos in wc_2026_matches.groupby('group'):
        times_do_grupo = dicionario_grupos_2026[grupo]

        tabela = {time: {'Grupo': grupo, 'Pts': 0, 'J': 0, 'V': 0, 'E': 0, 'D': 0, 'GM': 0, 'GS': 0, 'SG': 0} for time in times_do_grupo}

        for index, row in jogos.iterrows():
            tA = row['home_team']
            tB = row['away_team']
            gA = row['sim_gols_casa']
            gB = row['sim_gols_fora']

            tabela[tA]['J'] += 1
            tabela[tB]['J'] += 1
            tabela[tA]['GM'] += gA
            tabela[tB]['GM'] += gB
            tabela[tA]['GS'] += gB
            tabela[tB]['GS'] += gA

            if gA > gB:
                tabela[tA]['Pts'] += 3
                tabela[tA]['V'] += 1
                tabela[tB]['D'] += 1
            elif gA < gB:
                tabela[tB]['Pts'] += 3
                tabela[tB]['V'] += 1
                tabela[tA]['D'] += 1
            else:
                tabela[tA]['Pts'] += 1
                tabela[tB]['Pts'] += 1
                tabela[tA]['E'] += 1
                tabela[tB]['E'] += 1

        for time in times_do_grupo:
            tabela[time]['SG'] = tabela[time]['GM'] - tabela[time]['GS']

        df_grupo = pd.DataFrame.from_dict(tabela, orient='index')
        df_grupo = df_grupo.sort_values(by=['Pts', 'SG', 'GM'], ascending=[False, False, False])

        df_grupo['Posicao'] = range(1, 5)
        df_grupo['Time'] = df_grupo.index

        tabelas_grupos.append(df_grupo)

    df_classificacao_final = pd.concat(tabelas_grupos, ignore_index=True)

    classificados_diretos = df_classificacao_final[df_classificacao_final['Posicao'].isin([1, 2])]

    terceiros_colocados = df_classificacao_final[df_classificacao_final['Posicao'] == 3]

    melhores_terceiros = terceiros_colocados.sort_values(
        by=['Pts', 'SG', 'GM'],
        ascending=[False, False, False]
    ).head(8)

    times_mata_mata = pd.concat([classificados_diretos, melhores_terceiros]).sort_values(by=['Posicao'])

    chaveamento_estruturado = montar_chave_oficial_2026(times_mata_mata)

    fase_16_avos = jogar_fase(chaveamento_estruturado, dicionario_elos, modelo)
    oitavas      = jogar_fase(fase_16_avos, dicionario_elos, modelo)
    quartas      = jogar_fase(oitavas, dicionario_elos, modelo)
    semis        = jogar_fase(quartas, dicionario_elos, modelo)

    campeao = jogar_fase(semis, dicionario_elos, modelo)[0]
    campeoes.append(campeao)

chances_titulo = Counter(campeoes)
df_probabilidades = pd.DataFrame(chances_titulo.items(), columns=['Seleção', 'Títulos Simulados'])

df_probabilidades['Probabilidade (%)'] = (df_probabilidades['Títulos Simulados'] / simulacoes) * 100

df_probabilidades = df_probabilidades.sort_values(by='Probabilidade (%)', ascending=False).reset_index(drop=True)

print("\nCHANCES DE TÍTULO - COPA DO MUNDO 2026")
print(df_probabilidades.head(15))

Iniciando Monte Carlo: Simulando a Copa 100000 vezes...
CHANCES DE TÍTULO - COPA DO MUNDO 2026
          Seleção  Títulos Simulados  Probabilidade (%)
0          France              13437             13.437
1           Spain              13241             13.241
2       Argentina              13003             13.003
3         England              10023             10.023
4          Brazil               5093              5.093
5        Portugal               5018              5.018
6     Netherlands               4330              4.330
7         Belgium               4032              4.032
8         Morocco               3883              3.883
9         Germany               3361              3.361
10        Croatia               3246              3.246
11        Senegal               2768              2.768
12       Colombia               2585              2.585
13         Mexico               2149              2.149
14  United States               1986              1.986


## 14. Simulação de Confrontos Individuais de Mata-Mata
Criamos uma função para simular isoladamente um confronto eliminatório entre duas seleções específicas, incluindo a chance de o jogo ser decidido nos pênaltis, e a usamos para destacar os cenários Brasil x Japão e Brasil x Noruega.


In [59]:
def simular_jogo_mata_mata(time1, time2, modelo, num_simulacoes=100000):
    """
    Simula um confronto de mata-mata entre duas seleções usando o método de Monte Carlo.
    Também prevê o placar mais provável e sua porcentagem de confiança.
    """
    elo_t1 = elo[elo['country_full'] == time1]['total_points'].values[-1]
    elo_t2 = elo[elo['country_full'] == time2]['total_points'].values[-1]

    ataque_t1 = pd.DataFrame({
        'time': [time1], 'adversario': [time2],
        'mando_de_campo': [0], 'elo_time': elo_t1, 'elo_adversario': elo_t2
    })

    ataque_t2 = pd.DataFrame({
        'time': [time2], 'adversario': [time1],
        'mando_de_campo': [0], 'elo_time': elo_t2, 'elo_adversario': elo_t1
    })

    lambda_t1 = modelo.predict(ataque_t1).iloc[0]
    lambda_t2 = modelo.predict(ataque_t2).iloc[0]

    print(f"=======================================================")
    print(f"CONFRONTO: {time1.upper()} x {time2.upper()}")
    print(f"Gols Esperados: {time1} {lambda_t1:.2f} x {lambda_t2:.2f} {time2}")

    avancos_t1 = 0
    avancos_t2 = 0
    jogos_empatados = 0
    placares = []

    for _ in range(num_simulacoes):
        gols_t1 = np.random.poisson(lambda_t1)
        gols_t2 = np.random.poisson(lambda_t2)

        placares.append(f"{gols_t1} x {gols_t2}")

        if gols_t1 > gols_t2:
            avancos_t1 += 1
        elif gols_t2 > gols_t1:
            avancos_t2 += 1
        else:
            jogos_empatados += 1

            if np.random.random() < 0.5:
                avancos_t1 += 1
            else:
                avancos_t2 += 1

    prob_t1 = (avancos_t1 / num_simulacoes) * 100
    prob_t2 = (avancos_t2 / num_simulacoes) * 100
    prob_empate = (jogos_empatados / num_simulacoes) * 100
    
    contagem_placares = Counter(placares)
    placar_mais_comum, qtd_placar = contagem_placares.most_common(1)[0]
    confianca_placar = (qtd_placar / num_simulacoes) * 100

    print(f"--- Resultados após {num_simulacoes:,} Simulações ---")
    print(f"Placar Mais Provável: {time1} {placar_mais_comum} {time2} (Confiança: {confianca_placar:.1f}%)")
    print(f"Probabilidade de ir para os prorrogação/pênaltis: {prob_empate:.1f}%")
    print(f"Chances de {time1.upper()} avançar: {prob_t1:.1f}%")
    print(f"Chances de {time2.upper()} avançar: {prob_t2:.1f}%")
    print(f"=======================================================\n")


In [60]:
simular_jogo_mata_mata('Brazil', 'Japan', modelo)

CONFRONTO: BRAZIL x JAPAN
Gols Esperados: Brazil 1.29 x 0.86 Japan
--- Resultados após 100,000 Simulações ---
Placar Mais Provável: Brazil 1 x 0 Japan (Confiança: 15.1%)
Probabilidade de ir para os prorrogação/pênaltis: 28.7%
Chances de BRAZIL avançar: 60.8%
Chances de JAPAN avançar: 39.2%



## 15. Simulação 16 avos de final

In [62]:
simular_jogo_mata_mata('Germany', 'Paraguay', modelo)
simular_jogo_mata_mata('France', 'Sweden', modelo)
simular_jogo_mata_mata('South Africa', 'Canada', modelo)
simular_jogo_mata_mata('Netherlands', 'Morocco', modelo)
simular_jogo_mata_mata('Portugal', 'Croatia', modelo)
simular_jogo_mata_mata('Spain', 'Austria', modelo)
simular_jogo_mata_mata('United States', 'Bosnia and Herzegovina', modelo)
simular_jogo_mata_mata('Belgium', 'Senegal', modelo)

CONFRONTO: GERMANY x PARAGUAY
Gols Esperados: Germany 1.59 x 0.70 Paraguay
--- Resultados após 100,000 Simulações ---
Placar Mais Provável: Germany 1 x 0 Paraguay (Confiança: 16.1%)
Probabilidade de ir para os prorrogação/pênaltis: 25.1%
Chances de GERMANY avançar: 71.4%
Chances de PARAGUAY avançar: 28.6%

CONFRONTO: FRANCE x SWEDEN
Gols Esperados: France 2.13 x 0.52 Sweden
--- Resultados após 100,000 Simulações ---
Placar Mais Provável: France 2 x 0 Sweden (Confiança: 15.8%)
Probabilidade de ir para os prorrogação/pênaltis: 17.4%
Chances de FRANCE avançar: 83.9%
Chances de SWEDEN avançar: 16.1%

CONFRONTO: SOUTH AFRICA x CANADA
Gols Esperados: South Africa 0.84 x 1.33 Canada
--- Resultados após 100,000 Simulações ---
Placar Mais Provável: South Africa 0 x 1 Canada (Confiança: 15.3%)
Probabilidade de ir para os prorrogação/pênaltis: 28.3%
Chances de SOUTH AFRICA avançar: 37.4%
Chances de CANADA avançar: 62.6%

CONFRONTO: NETHERLANDS x MOROCCO
Gols Esperados: Netherlands 1.09 x 1.01 Mor

In [63]:
simular_jogo_mata_mata('Brazil', 'Japan', modelo)
simular_jogo_mata_mata('Ivory Coast', 'Norway', modelo)
simular_jogo_mata_mata('Mexico', 'Ecuador', modelo)
simular_jogo_mata_mata('England', 'DR Congo', modelo)
simular_jogo_mata_mata('Argentina', 'Cape Verde', modelo)
simular_jogo_mata_mata('Australia', 'Egypt', modelo)
simular_jogo_mata_mata('Switzerland', 'Algeria', modelo)
simular_jogo_mata_mata('Colombia', 'Ghana', modelo)

CONFRONTO: BRAZIL x JAPAN
Gols Esperados: Brazil 1.29 x 0.86 Japan
--- Resultados após 100,000 Simulações ---
Placar Mais Provável: Brazil 1 x 0 Japan (Confiança: 15.0%)
Probabilidade de ir para os prorrogação/pênaltis: 28.7%
Chances de BRAZIL avançar: 61.0%
Chances de JAPAN avançar: 39.0%

CONFRONTO: IVORY COAST x NORWAY
Gols Esperados: Ivory Coast 1.00 x 1.12 Norway
--- Resultados após 100,000 Simulações ---
Placar Mais Provável: Ivory Coast 0 x 1 Norway (Confiança: 13.6%)
Probabilidade de ir para os prorrogação/pênaltis: 29.6%
Chances de IVORY COAST avançar: 47.1%
Chances de NORWAY avançar: 52.9%

CONFRONTO: MEXICO x ECUADOR
Gols Esperados: Mexico 1.23 x 0.90 Ecuador
--- Resultados após 100,000 Simulações ---
Placar Mais Provável: Mexico 1 x 0 Ecuador (Confiança: 14.8%)
Probabilidade de ir para os prorrogação/pênaltis: 28.9%
Chances de MEXICO avançar: 58.4%
Chances de ECUADOR avançar: 41.6%

CONFRONTO: ENGLAND x DR CONGO
Gols Esperados: England 2.06 x 0.53 DR Congo
--- Resultados ap